# LLM Generation Parameters

## Import Libraries

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [2]:
# Always remember to do this! - What does this do?
load_dotenv(override=True)

False

## Exploring Parameters

### OpenAI Chat Completions API (most widely used)

`OpenAI` Official documentation is here - https://platform.openai.com/docs/api-reference/chat/create Refer to this for a full set of parameters that we'll later explore. 

The most used parameters are as listed below, with definitions from official `OpenAI` API documentation. These are considered "common" because they fundamentally control how an autoregressive language model samples tokens to generate output. 

- **max_tokens** (only *max_completion_tokens* is supported by o1 series; no default): The maximum number of completion tokens that may be used over the course of the run. The run will make a best effort to use only the number of completion tokens specified, across multiple turns of the run. If the run exceeds the number of completion tokens specified, the run will end with status `incomplete`.
- **temperature** (*1 by default*): What sampling temperature to use, between 0 and 2. Higher values like 0.8 will make the output more random, while lower values like 0.2 will make it more focused and deterministic. We generally recommend altering this or top_p but not both.
- **top_p** (*1 by default*): An alternative to sampling with temperature, called nucleus sampling, where the model considers the results of the tokens with top_p probability mass. So 0.1 means only the tokens comprising the top 10% probability mass are considered.
We generally recommend altering this or temperature but not both.
- **top_k**: not available in OpenAI client library
- **frequency_penalty** (*0 by default*): Number between -2.0 and 2.0. Positive values penalize new tokens based on their existing frequency in the text so far, decreasing the model's likelihood to repeat the same line verbatim.
- **presence_penalty** (*0 by default*): Number between -2.0 and 2.0. Positive values penalize new tokens based on whether they appear in the text so far, increasing the model's likelihood to talk about new topics.
- **stop** (*null by default*): Not supported with latest reasoning models o3 and o4-mini. Up to 4 sequences where the API will stop generating further tokens. The returned text will not contain the stop sequence.

### Anthropic's Messages API

It's good to note that Anthropic's `Messages` API contains similar parameters except a few differences:

- **max_tokens** is used as is
- **temperature** has a value between 0 and 1
- **top_p** is as is
- **top_k** is also available: Only sample from the top K options for each subsequent token. Used to remove "long tail" low probability responses. Recommended for advanced use cases only. You usually only need to use temperature.
- **frequency_penalty** and **presence_penalty** parameters are not available.
- **stop_sequences** is used instead of **stop**

### Some important considerations

- Exact naming conventions of parameters can vary, but functionality is same (like we saw with Anthropic's API).
- The optimal range or impact of a parameter might differ between models, even if general principle is same. A temperature of 1.0 might be very wild for one model but only moderately creative for another.
- Differences between different parameter configurations will be more prominent in larger LLMs, as with smaller LLMs, the "spread" of probabilities for the next token might not be as wide.

### Setting temperature vs. top_p for different use cases: 

| Use Case               | Temperature | Top_p | Description                                                                                        |
| :--------------------- | :---------- | :---- | :------------------------------------------------------------------------------------------------- |
| Code Generation        | 0.2         | 0.1   | Generates code that adheres to established patterns and conventions. Output is more deterministic and focused. Useful for generating syntactically correct code. |
| Creative Writing       | 0.7         | 0.8   | Generates creative and diverse text for storytelling. Output is more exploratory and less constrained by patterns. |
| Chatbot Responses      | 0.5         | 0.5   | Generates conversational responses that balance coherence and diversity. Output is more natural and engaging. |
| Code Comment Generation | 0.3         | 0.2   | Generates code comments that are more likely to be concise and relevant. Output is more deterministic and adheres to conventions. |
| Data Analysis Scripting | 0.2         | 0.1   | Generates data analysis scripts that are more likely to be correct and efficient. Output is more deterministic and focused. |
| Exploratory Code Writing | 0.6         | 0.7   | Generates code that explores alternative solutions and creative approaches. Output is less constrained by established patterns. |

References from here: https://community.openai.com/t/cheat-sheet-mastering-temperature-and-top-p-in-chatgpt-api/172683/10

### Setting Up

In [5]:
# Use either Llama3.2 (3B) with Ollama
!ollama pull llama3.2    

# Initialize the OpenAI client to connect to Ollama
llm_api = OpenAI(base_url='http://localhost:11434/v1', 
                 api_key='ollama') # note that llm_api is a good variable name, because the LLM is indeed exposed locally as an API by Ollama!

# Define the model to use
model_name = "llama3.2" # Ensure this matches the pulled model in Ollama

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [4]:
# # Or use DeepSeek-R1-Distill-Llama-70B from Groq
# llm_api = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")

# # Define the model to use
# model_name = "deepseek-r1-distill-llama-70b"

In [6]:
#Helper function for text generation
def generate_and_display(prompt, param_name, param_value, num_runs=1, **kwargs):
    """
    Helper function to make a call to the LLM and display results.
    Runs multiple times for stochastic parameters to highlight differences.
    """
    print(f"\n--- Demonstrating {param_name}={param_value} ---")
    print(f"Prompt: \"{prompt}\"")
    print(f"Other consistent parameters: {kwargs}")

    messages = [
        {"role": "system", "content": "You are a helpful and concise AI assistant for a business."},
        {"role": "user", "content": prompt},
    ]

    for i in range(num_runs):
        print(f"\n--- Run {i+1} ---")
        try:
            response = llm_api.chat.completions.create(
                model=model_name,
                messages=messages,
                **{param_name: param_value, **kwargs}
            )
            print("Generated Text:")
            answer = response.choices[0].message.content.strip()
            display(Markdown(answer)) # .strip() removes leading/trailing whitespace for cleaner output
        except Exception as e:
            print(f"An error occurred: {e}")
            print("Ensure Ollama is running and the model is pulled. Skipping further runs for this parameter setting.")
            break # Exit loop if an error occurs

    print("\n--- End Demonstration for this parameter setting ---\n" + "="*80)

### 1. Temperature

**Commercial Use Cases**: Content generation (marketing copy, ad variations, blog post ideas), creative brainstorming for product features, unique customer engagement messages.

**What to expect:**
- **Low Temperature**: Look for very similar, factual, and direct outputs across runs. Less "fluff," more to the point.
    - E.g., Consistent, factual product bullet points for a catalog
- **Medium Temperature**: Outputs will start showing more natural language, some variation in phrasing, but still coherent.
    - E.g., Engaging marketing copy for a landing page
- **High Temperature**: Expect more varied vocabulary, potentially more metaphorical or unusual phrasing, and noticeable differences between runs. It might sometimes deviate slightly from the core prompt if pushed too high.
    - E.g., Brainstorming unique ad slogans or creative concepts for a campaign

In [7]:
common_prompt_temp = """
Merck is heavily investing in Agentic AI and AI Gateways to personalize healthcare for millions of patients. 
Envision a new Agentic AI-powered service, "Merck Vitals," that leverages patient data and medical knowledge.
First, describe the service focusing on its innovative features and how it will enhance daily life for a typical user.
Then, think about the role of AI gateway in enabling a production-ready solution.
Keep the answer less than 70 tokens."""

In [8]:
# Example 1.1: Low Temperature
generate_and_display(
    common_prompt_temp,
    "temperature", 0.1,
    num_runs=3, # Run multiple times to show consistency
    max_completion_tokens=70
)


--- Demonstrating temperature=0.1 ---
Prompt: "
Merck is heavily investing in Agentic AI and AI Gateways to personalize healthcare for millions of patients. 
Envision a new Agentic AI-powered service, "Merck Vitals," that leverages patient data and medical knowledge.
First, describe the service focusing on its innovative features and how it will enhance daily life for a typical user.
Then, think about the role of AI gateway in enabling a production-ready solution.
Keep the answer less than 70 tokens."
Other consistent parameters: {'max_completion_tokens': 70}

--- Run 1 ---
Generated Text:


**Merck Vitals: Personalized Healthcare Service**

Merck Vitals is an Agentic AI-powered service that empowers patients to take control of their health. Key features include:

* **Personalized Health Profiles**: AI-driven analysis of patient data, medical history, and lifestyle habits.
* **Predictive Insights**: Proactive alerts for potential health risks and preventive measures.
* **Medication Optimization**: AI-recommended treatment plans and dosage adjustments.
* **Virtual Health Coaching**: AI-powered guidance for healthy habits and wellness.

**AI Gateway Enabling Solution**

The AI gateway plays a crucial role in enabling a production-ready solution by:

* **Integrating Patient Data**: Seamlessly connecting patient data from various sources.
* **Standardizing Medical Knowledge**: Leveraging Merck's vast medical knowledge to inform AI decision-making.
* **Ensuring Data Security**: Robust security measures to protect sensitive patient information.

By combining these features, Merck Vitals revolutionizes patient care, empowering individuals to make informed decisions about their health.


--- Run 2 ---
Generated Text:


**Merck Vitals: Personalized Healthcare Service**

Merck Vitals is an Agentic AI-powered service that empowers patients to take control of their health. Key features include:

* AI-driven health risk assessments and personalized recommendations
* Integration with wearable devices and electronic health records
* Virtual consultations with AI-assisted doctors for informed decision-making
* Real-time monitoring and alerts for timely interventions

**AI Gateway Enabling Production-Ready Solution**

The AI gateway plays a crucial role in enabling Merck Vitals by:

* Securing and standardizing patient data for seamless integration
* Providing a scalable and reliable infrastructure for AI model deployment
* Ensuring compliance with regulatory requirements and industry standards
* Facilitating seamless communication between AI systems and healthcare providers


--- Run 3 ---
Generated Text:


**Merck Vitals: Personalized Healthcare Service**

Merck Vitals is an Agentic AI-powered service that empowers patients to take control of their health. Key features include:

* **Personalized Health Profiles**: AI-driven analysis of patient data, medical history, and lifestyle habits to create tailored recommendations.
* **Predictive Analytics**: AI-powered predictions of potential health risks and preventive measures to mitigate them.
* **Virtual Health Coaching**: AI-driven guidance on healthy habits, nutrition, and wellness strategies.

**AI Gateway Enabling Production-Ready Solution**

The AI gateway plays a crucial role in integrating patient data, medical knowledge, and AI algorithms to create a seamless and secure experience. It enables:

* **Data Integration**: Consolidates patient data from various sources, ensuring accuracy and completeness.
* **Security and Compliance**: Ensures data protection and adherence to regulatory standards.
* **Scalability and Flexibility**: Supports the growth of the service, allowing for easy expansion and customization.

By leveraging the AI gateway, Merck Vitals can provide a robust, secure, and scalable solution for personalized healthcare, enhancing daily life for millions of patients.


--- End Demonstration for this parameter setting ---


In [9]:
# Example 1.2: Medium Temperature
generate_and_display(
    common_prompt_temp,
    "temperature", 0.7,
    num_runs=3, # Run multiple times to show controlled variation
    max_completion_tokens=70
)


--- Demonstrating temperature=0.7 ---
Prompt: "
Merck is heavily investing in Agentic AI and AI Gateways to personalize healthcare for millions of patients. 
Envision a new Agentic AI-powered service, "Merck Vitals," that leverages patient data and medical knowledge.
First, describe the service focusing on its innovative features and how it will enhance daily life for a typical user.
Then, think about the role of AI gateway in enabling a production-ready solution.
Keep the answer less than 70 tokens."
Other consistent parameters: {'max_completion_tokens': 70}

--- Run 1 ---
Generated Text:


**Merck Vitals: Personalized Healthcare Service**

Merck Vitals is an Agentic AI-powered service that leverages patient data and medical knowledge to provide personalized health recommendations. Key features include:

* AI-driven health risk assessments and personalized action plans
* Integration with wearable devices and medical records
* Real-time access to expert advice and support

**AI Gateway Enablement**

The AI gateway enables a production-ready solution by:

* Simplifying data integration and standardization
* Ensuring data security and compliance
* Facilitating seamless API connections for advanced analytics and insights

By leveraging the AI gateway, Merck Vitals can provide a scalable and reliable platform for delivering personalized healthcare services to millions of patients.


--- Run 2 ---
Generated Text:


**Merck Vitals: Personalized Healthcare Service**

Merck Vitals is an Agentic AI-powered service that empowers patients to take control of their health. Key features include:

* AI-driven personalized health recommendations based on individual needs and medical history
* Integrated medication management and adherence tracking
* Virtual health coaching and emotional support from certified healthcare professionals
* Seamless integration with wearable devices and electronic health records (EHRs)

**AI Gateway Integration**

AI gateways play a crucial role in enabling a production-ready solution for Merck Vitals. They facilitate:

* Secure data exchange between devices, EHRs, and the cloud
* Real-time processing and analysis of patient data
* Scalable and flexible architecture for seamless integration with various healthcare systems
* End-to-end data encryption and security measures to protect patient confidentiality

By leveraging AI gateways, Merck Vitals can deliver a secure, scalable, and personalized healthcare experience to millions of patients worldwide.


--- Run 3 ---
Generated Text:


**Merck Vitals:**

Merck Vitals is an Agentic AI-powered service that revolutionizes personalized healthcare. Key features include:

* AI-driven health risk assessments and tailored recommendations
* Personalized medication management and adherence tracking
* Integration with wearable devices and EHRs for seamless data exchange

**Enhancing daily life:**

Merck Vitals empowers users to take control of their health, making informed decisions and achieving better health outcomes. With AI-driven insights and recommendations, users can:

* Identify potential health risks and preventive measures
* Optimize medication regimens for maximum efficacy and safety
* Monitor progress and receive personalized support

**AI Gateway Role:**

The AI Gateway plays a crucial role in enabling a production-ready solution by:

* Providing a scalable and secure data infrastructure
* Integrating with various medical devices, EHRs, and wearables
* Enabling real-time data processing and analytics for actionable insights

By harnessing the power of AI Gateways, Merck Vitals can provide a comprehensive, user-centric platform for personalized healthcare.


--- End Demonstration for this parameter setting ---


In [10]:
# Example 1.3: High Temperature
generate_and_display(
    common_prompt_temp,
    "temperature", 1.5, # Pushing temperature higher for more dramatic effect
    num_runs=3, # Run multiple times to show increased randomness
    max_completion_tokens=70
)


--- Demonstrating temperature=1.5 ---
Prompt: "
Merck is heavily investing in Agentic AI and AI Gateways to personalize healthcare for millions of patients. 
Envision a new Agentic AI-powered service, "Merck Vitals," that leverages patient data and medical knowledge.
First, describe the service focusing on its innovative features and how it will enhance daily life for a typical user.
Then, think about the role of AI gateway in enabling a production-ready solution.
Keep the answer less than 70 tokens."
Other consistent parameters: {'max_completion_tokens': 70}

--- Run 1 ---
Generated Text:


Merck Vitals is an Agentic AI-powered service that integrates personalized healthcare into daily life. Novel features include:

Virtual Nurse Navigation: AI-assisted personalized care plans, automating routine inquiries and follow-up requests.
Genomic Guidance: Patient genetic profiles matched with recommended treatments, providing empowering insights.
Personalized Prescription Refills: Integrated electronic health records (EHR) enable AI-optimized refill requests.

AI Gateway enables production-ready solution by harnessing scalable and secure infrastructure, enabling:

Seamless Interoperability: Access to aggregated patient data across healthcare ecosystems.
Dynamic Configuration: Real-time configuration of AI models, ensuring personalized recommendations and optimal treatment outcomes.
Multi-Environment Support: Scalability for expanding services across care settings, empowering users with comprehensive support ecosystem.


--- Run 2 ---
Generated Text:


"Merck Vitals" is an Agentic AI-powered service that utilizes patient data and medical knowledge to personalize healthcare. Key features include:

* AI-driven symptom tracking and analysis
* Personalized treatment recommendations from a knowledgeable AI assistant
* Integration with wearables, medical records, and appointment scheduling

Benefits for a typical user:

* Enhanced care experience through tailored recommendations and proactive health monitoring
* Convenience with automations and reminders

AI Gateway: A critical layer, providing a platform for scalability, security, and standardization, allowing multiple external integrations and ensuring seamless data flow for AI model training and continuous improvement.


--- Run 3 ---
Generated Text:


'Merck Vitals' is an Agentic AI-powered service that empowers patients to take control of their health. Innovative features include: AI-driven health risk assessments, personalized treatment plans, and proactive disease prevention alerts. The service learns from user feedback and adapting to individual needs.

The AI gateway enables seamless integration of patient data, linking it with aggregated medical knowledge. Secure APIs ensure data protection while facilitating data-driven innovation, and scalable backend architecture ensures production-readiness for mass adaptation across medical communities.


--- End Demonstration for this parameter setting ---


### 2. Max_tokens

**Commercial Use Cases**: Chatbot response length control, email subject line generation, tweet generation, summarizing lengthy documents for internal reports, generating specific-length ad copy.

**What to expect**: 

- The generated output will strictly adhere to the `max_tokens` limit, regardless of whether the summary is complete.
- Depending on the use case, the `max_tokens` parameter can be set:
    - Very few tokens (e.g., for a headline or quick alert)
    - Moderate tokens (e.g., for an executive summary or internal Slack message)
    - More tokens (e.g., for a detailed summary section in a newsletter or report)

In [11]:
common_prompt_max_tokens = """
Imagine you're a data scientist working with an IPL team. 
Describe how Generative AI could be used to create novel training drills or strategic simulations for batsmen, considering the unique challenges of Indian pitches and varied bowling styles."""

In [12]:
# Example 2.1: Very few tokens
generate_and_display(
    common_prompt_max_tokens + " Complete response in less than 15 tokens.",
    "max_tokens", 15,
    temperature=0.5
)


--- Demonstrating max_tokens=15 ---
Prompt: "
Imagine you're a data scientist working with an IPL team. 
Describe how Generative AI could be used to create novel training drills or strategic simulations for batsmen, considering the unique challenges of Indian pitches and varied bowling styles. Complete response in less than 15 tokens."
Other consistent parameters: {'temperature': 0.5}

--- Run 1 ---
Generated Text:


As a data scientist for an IPL team, I'd leverage Generative AI


--- End Demonstration for this parameter setting ---


In [13]:
# Example 2.2: Moderate tokens
generate_and_display(
    common_prompt_max_tokens,
    "max_tokens", 50,
    temperature=0.5
)


--- Demonstrating max_tokens=50 ---
Prompt: "
Imagine you're a data scientist working with an IPL team. 
Describe how Generative AI could be used to create novel training drills or strategic simulations for batsmen, considering the unique challenges of Indian pitches and varied bowling styles."
Other consistent parameters: {'temperature': 0.5}

--- Run 1 ---
Generated Text:


As a data scientist working with an IPL team, I'd like to propose the use of Generative AI to create novel training drills and strategic simulations for batsmen, taking into account the unique challenges of Indian pitches and varied bowling styles.

**Generative


--- End Demonstration for this parameter setting ---


In [14]:
# Example 2.3: More tokens
generate_and_display(
    common_prompt_max_tokens,
    "max_tokens", 150,
    temperature=0.5
)


--- Demonstrating max_tokens=150 ---
Prompt: "
Imagine you're a data scientist working with an IPL team. 
Describe how Generative AI could be used to create novel training drills or strategic simulations for batsmen, considering the unique challenges of Indian pitches and varied bowling styles."
Other consistent parameters: {'temperature': 0.5}

--- Run 1 ---
Generated Text:


As a data scientist working with an IPL team, I'd like to explore how Generative AI can revolutionize training drills and strategic simulations for batsmen, taking into account the unique challenges of Indian pitches and varied bowling styles.

**Training Drills:**

1. **Pitch-Specific Drills:** Utilize Generative AI to generate customized training drills tailored to specific pitches, such as the high-bouncing tracks of the UAE or the slower, more spin-friendly surfaces of India. The AI can analyze historical data on batsmen's performance on these pitches and generate drills that mimic the conditions, helping batsmen prepare for the challenges they'll face.
2. **Bowling Style-Specific Drills:** Create simulations that mimic the unique bowling styles of individual


--- End Demonstration for this parameter setting ---


In [15]:
#Example 2.4: Even more max_tokens
generate_and_display(
    common_prompt_max_tokens,
    "max_tokens", 1000,
    temperature=0.5
)


--- Demonstrating max_tokens=1000 ---
Prompt: "
Imagine you're a data scientist working with an IPL team. 
Describe how Generative AI could be used to create novel training drills or strategic simulations for batsmen, considering the unique challenges of Indian pitches and varied bowling styles."
Other consistent parameters: {'temperature': 0.5}

--- Run 1 ---
Generated Text:


As a data scientist working with an IPL team, I'd like to explore the potential of Generative AI (GA) in creating novel training drills and strategic simulations for batsmen. Here's a possible approach:

**Data Collection**

To train the GA model, we would need a vast amount of data on:

1. **Pitch conditions**: Historical data on India's diverse pitches, including factors like spin, pace, and bounce.
2. **Bowling styles**: Data on various bowling techniques, including spinners, fast bowlers, and all-rounders.
3. **Batting performances**: Historical data on batsmen's performances against different bowlers, pitches, and conditions.
4. **Drill and simulation data**: Data on existing training drills and simulations used by the team, including parameters like bat speed, swing rate, and shot selection.

**Generative AI Model**

We would train a GA model using the collected data to generate novel training drills and simulations. The model could be based on techniques like:

1. **Recurrent Neural Networks (RNNs)**: To learn patterns and relationships between pitch conditions, bowling styles, and batting performances.
2. **Generative Adversarial Networks (GANs)**: To create novel, realistic simulations of bowling and batting scenarios.
3. **Variational Autoencoders (VAEs)**: To compress and generate data on training drills and simulations, allowing for more efficient exploration of the solution space.

**Training Drill and Simulation Generation**

The GA model would be trained to generate new training drills and simulations that:

1. **Mitigate weaknesses**: Identify and address specific weaknesses in the batsmen's technique, such as footwork, bat speed, or shot selection.
2. **Enhance strengths**: Amplify the batsmen's strengths, such as their ability to play late or their power-hitting skills.
3. **Adapt to different conditions**: Generate drills and simulations that simulate various pitch conditions, such as spin, pace, and bounce.
4. **Introduce new challenges**: Create novel, unpredictable scenarios that test the batsmen's decision-making and reaction time.

**Strategic Simulation**

The GA model could also be used to generate strategic simulations that:

1. **Assess team strengths and weaknesses**: Evaluate the team's overall performance, identifying areas of improvement and potential vulnerabilities.
2. **Predict opponent behavior**: Simulate the behavior of opposing teams, including their batting and bowling strategies, to inform the team's own strategy.
3. **Optimize batting lineups**: Generate batting lineups that balance individual strengths and weaknesses, taking into account factors like pitch conditions and bowling styles.
4. **Develop game plans**: Create game plans that adapt to different scenarios, such as chasing down targets or defending scores.

**Implementation and Evaluation**

To implement the GA model, we would:

1. **Integrate with existing training systems**: Integrate the GA model with the team's existing training systems, allowing for seamless interaction and feedback.
2. **Monitor and evaluate performance**: Continuously monitor the batsmen's performance and evaluate the effectiveness of the generated training drills and simulations.

By leveraging Generative AI, we can create novel training drills and strategic simulations that help the batsmen improve their skills, adapt to different conditions, and make informed decisions during games.


--- End Demonstration for this parameter setting ---


### 3. Top_p (Nucleus sampling)

**Commercial Use Cases**: Generating diverse customer service responses (without going off-topic), creating varied product names, generating slightly different content variations for A/B testing.

**What to Observe**:

- **Low** top_p: Outputs will be very similar across runs, often picking the most statistically common words/phrases. Less variation in phrasing.

- **Medium** top_p: Some variations in phrasing will emerge, but still within a coherent and expected range.

- **High** top_p: More diverse and potentially more "creative" word choices and phrasing, showing greater variation across runs.

In [16]:
common_prompt_top_p = """
A customer is asking about our new premium coffee blend, that rivals Starbucks, Peets Coffee and Cafe Coffee Day. 
Provide a welcoming response that also highlights its unique flavor notes."""

In [17]:
# Example 3.1: Low top_p (for a very standard, consistent customer service greeting)
generate_and_display(
    common_prompt_top_p,
    "top_p", 0.1,
    num_runs=3,
    temperature=0.7,
    max_tokens=80
)


--- Demonstrating top_p=0.1 ---
Prompt: "
A customer is asking about our new premium coffee blend, that rivals Starbucks, Peets Coffee and Cafe Coffee Day. 
Provide a welcoming response that also highlights its unique flavor notes."
Other consistent parameters: {'temperature': 0.7, 'max_tokens': 80}

--- Run 1 ---
Generated Text:


"Welcome to our coffee shop! We're thrilled to introduce our newest premium coffee blend, carefully crafted to rival the best of the best. Our expert roasters have worked tirelessly to create a truly exceptional blend that's sure to tantalize your taste buds.

Our premium coffee blend is a masterful blend of Arabica beans from around the world, carefully selected for their rich, full-bodied flavor and unique


--- Run 2 ---
Generated Text:


"Welcome to our coffee shop! We're thrilled to introduce our newest premium coffee blend, carefully crafted to rival the best of the best. Our expert roasters have worked tirelessly to create a truly exceptional blend that's sure to tantalize your taste buds.

Our premium coffee blend is a masterful blend of Arabica beans from around the world, carefully selected for their rich, full-bodied flavor and unique


--- Run 3 ---
Generated Text:


"Welcome to our coffee shop! We're thrilled to introduce our newest premium coffee blend, carefully crafted to rival the best of the best. Our expert roasters have worked tirelessly to create a truly exceptional blend that's sure to tantalize your taste buds.

Our premium coffee blend is a masterful blend of Arabica beans from around the world, carefully selected for their rich, full-bodied flavor and unique


--- End Demonstration for this parameter setting ---


In [18]:
# Example 3.2: Medium top_p (for a friendly, slightly varied customer service interaction)
generate_and_display(
    common_prompt_top_p,
    "top_p", 0.5,
    num_runs=3,
    temperature=0.7,
    max_tokens=80
)


--- Demonstrating top_p=0.5 ---
Prompt: "
A customer is asking about our new premium coffee blend, that rivals Starbucks, Peets Coffee and Cafe Coffee Day. 
Provide a welcoming response that also highlights its unique flavor notes."
Other consistent parameters: {'temperature': 0.7, 'max_tokens': 80}

--- Run 1 ---
Generated Text:


"Thank you for considering our premium coffee blend! We're thrilled to introduce our newest offering, carefully crafted to rival the best of the industry. Our unique blend is a masterful combination of expertly sourced Arabica beans from around the world, carefully roasted to bring out the full depth of flavor.

What sets our blend apart is its distinctive flavor profile, which features notes of rich chocolate, hints of


--- Run 2 ---
Generated Text:


Here's a welcoming response that highlights the unique flavor notes of your premium coffee blend:

"Thank you for considering our premium coffee blend! We're thrilled to introduce our newest offering, carefully crafted to rival the best of the best in the coffee world. Our expert roasters have carefully selected a blend of rare and exotic beans from around the world to create a truly unique flavor profile.

Our premium coffee blend


--- Run 3 ---
Generated Text:


"Welcome to our coffee shop! We're thrilled to introduce our newest premium coffee blend, carefully crafted to rival the best of the industry. Our expert roasters have carefully selected a blend of the finest Arabica beans from around the world, resulting in a truly unique flavor profile.

Our premium coffee blend boasts a rich, smooth taste with hints of dark chocolate, caramel, and a subtle hint of fruit


--- End Demonstration for this parameter setting ---


In [19]:
# Example 3.3: High top_p (for brainstorming diverse opening lines for marketing outreach)
generate_and_display(
    common_prompt_top_p,
    "top_p", 0.99, # Pushing higher for more noticeable effect
    num_runs=3,
    temperature=0.7,
    max_tokens=80
)


--- Demonstrating top_p=0.99 ---
Prompt: "
A customer is asking about our new premium coffee blend, that rivals Starbucks, Peets Coffee and Cafe Coffee Day. 
Provide a welcoming response that also highlights its unique flavor notes."
Other consistent parameters: {'temperature': 0.7, 'max_tokens': 80}

--- Run 1 ---
Generated Text:


"Welcome to our coffee shop! I'm thrilled to hear that you're interested in our new premium coffee blend. We're incredibly proud of this unique offering, which has been carefully crafted to rival the best of the industry.

Our premium coffee blend is a masterful blend of expertly roasted Arabica beans, sourced from the world's top coffee-producing regions. What sets it apart, however, is


--- Run 2 ---
Generated Text:


Here's a welcoming response that highlights the unique flavor notes of your premium coffee blend:

"Hello and thank you for considering our premium coffee blend! We're thrilled to introduce you to our latest creation, carefully crafted to rival the best of the industry - Starbucks, Peet's Coffee, and Café Coffee Day.

Our premium blend is a masterful blend of expertly sourced Arabica beans, roasted to


--- Run 3 ---
Generated Text:


"Welcome to our coffee corner! We're thrilled to introduce our newest premium coffee blend, which has been generating a buzz among coffee connoisseurs. Our expertly crafted blend is indeed a force to be reckoned with, rivaling the best of Starbucks, Peets Coffee, and Cafe Coffee Day.

What sets our blend apart is its unique flavor profile, which showcases a rich and smooth taste


--- End Demonstration for this parameter setting ---


### 4. Frequency_penalty

**Commercial Use Cases**: Ensuring variety in marketing emails, preventing chatbots from repeating FAQs, generating diverse social media posts about the same product.

**What to Observe**:

- **No** penalty (0.0): Look for words or short phrases being repeated frequently within the generated paragraph.

- **Moderate** penalty: Repetition should be noticeably reduced, leading to more varied sentences.

- **High** penalty: The model will actively avoid repeating words, potentially leading to more complex or less natural phrasing if it has to find many synonyms.

In [20]:
common_prompt_freq_penalty = """
Give me 15 different phrases that customer support executives often say. Only return the phrases.
"""

In [21]:
# Example 4.1: Negative frequency penalty (encourages repetition)
generate_and_display(
    common_prompt_freq_penalty,
    "frequency_penalty", -1,
    temperature=0.7,
    seed = 42
)


--- Demonstrating frequency_penalty=-1 ---
Prompt: "
Give me 15 different phrases that customer support executives often say. Only return the phrases.
"
Other consistent parameters: {'temperature': 0.7, 'seed': 42}

--- Run 1 ---
Generated Text:


1. "I'd be happy to assist you further."
2. "Can you please provide more details?"
3. "I apologize for the inconvenience."
4. "I'd like to offer you a solution."
5. "I'm here to help you."
6. "I'd be happy to help you."
7. "I'd like to escalate this to a supervisor."
8. "I apologize for the delay."
9. "I'm doing my best to resolve."
10. "I'd like to provide you with a quote."
11. "I'd like to schedule a follow-up."
12. "I'd like to check on the status."
13. "I'd like to confirm."
14. "I'd like to resolve."
15. "I'd like to close the ticket."


--- End Demonstration for this parameter setting ---


In [22]:
# Example 4.2: Moderate frequency penalty (reduced repetition)
generate_and_display(
    common_prompt_freq_penalty,
    "frequency_penalty", 1.0, # Increased for more impact
    temperature=0.7,
    seed=42
)


--- Demonstrating frequency_penalty=1.0 ---
Prompt: "
Give me 15 different phrases that customer support executives often say. Only return the phrases.
"
Other consistent parameters: {'temperature': 0.7, 'seed': 42}

--- Run 1 ---
Generated Text:


1. "I'd be happy to assist you further."
2. "Can you please provide more details?"
3. "I apologize for the inconvenience."
4. "Let me check on that for you."
5. "How can I help you today?"
6. "I'll do my best to resolve this issue."
7. "Can I offer an alternative solution?"
8. "What would be your preferred outcome?"
9. "I'm happy to escalate this issue if needed."
10. "Can you confirm your order details?"
11. "How did you experience the problem?"
12. "I'm here to help, what's on your mind?"
13. "Is there anything else I can assist you with today?"
14. "Can I provide a refund/exchange for you?"
15. "Would you like me to explain the process in more detail?"


--- End Demonstration for this parameter setting ---


In [23]:
# Example 4.3: High frequency penalty (strong reduction in repetition, might affect coherence)
generate_and_display(
    common_prompt_freq_penalty,
    "frequency_penalty", 2.0, # Max penalty for strong effect
    temperature=0.7,
    seed=42
)


--- Demonstrating frequency_penalty=2.0 ---
Prompt: "
Give me 15 different phrases that customer support executives often say. Only return the phrases.
"
Other consistent parameters: {'temperature': 0.7, 'seed': 42}

--- Run 1 ---
Generated Text:


1. "I'd be happy to assist you further."
2. "Can you please provide more details?"
3. "Let me see what I can do."
4. "I apologize for the inconvenience caused."
5. "What's your order number?"
6. "Please stand by for just a moment."
7."Would you like me to escalate this issue?"
8."I've checked on that and it appears..."
9."Can you confirm your shipping address?"
10. "Your account information will be updated shortly."
11. "Have you tried turning it off and on again?"
12. "I'd like to offer a solution for this issue."
13. "Please allow me to check our records."
14.""Is everything okay with your purchase today?"
15."I'll do my best to assist you personally."


--- End Demonstration for this parameter setting ---


### 5. Presence_penalty

**Commercial Use Cases**: Encouraging a customer service bot to explore different solutions/topics, generating comprehensive meeting minutes by covering all points, producing diverse content ideas for a campaign.

**What to Observe**:

- **No** penalty (0.0): The model might dwell on a few initial topics or points without moving on to others mentioned in the conceptual prompt.

- **Moderate** penalty: The summary should cover a broader range of distinct action items or concepts, rather than just elaborating on the first few.

- **High** penalty: The model will try hard to introduce new concepts or distinct ideas, possibly jumping between points quickly or even generating somewhat disjointed output if forced too much.

In [24]:
common_prompt_pres_penalty = """
You are leading a quick internal meeting discussing how Generative AI can significantly improve our current sales and customer engagement processes. 
Briefly outline 4 distinct areas or strategies where GenAI could have a major impact, providing a brief example for each.
"""

In [25]:
# Example 5.1: No presence penalty (might focus heavily on one or two main themes)
generate_and_display(
    common_prompt_pres_penalty,
    "presence_penalty", 0.0,
    temperature=0.7,
)


--- Demonstrating presence_penalty=0.0 ---
Prompt: "
You are leading a quick internal meeting discussing how Generative AI can significantly improve our current sales and customer engagement processes. 
Briefly outline 4 distinct areas or strategies where GenAI could have a major impact, providing a brief example for each.
"
Other consistent parameters: {'temperature': 0.7}

--- Run 1 ---
Generated Text:


Good morning, team. Today, we're discussing the potential of Generative AI to boost our sales and customer engagement processes. After reviewing the market and our current processes, I'd like to highlight four distinct areas where GenAI could have a major impact:

1. **Personalized Sales Content**: GenAI can generate customized sales content, such as product descriptions, sales scripts, and even entire pitches, in real-time based on customer needs and preferences. For example, our marketing team can input customer data, and the GenAI system will produce a personalized sales script that adjusts to the customer's industry, job function, and pain points.

2. **Chatbot-Driven Customer Support**: GenAI-powered chatbots can handle routine customer inquiries, freeing up our human support agents to focus on more complex issues. For instance, we can train the chatbot to recognize common customer questions and respond with pre-written answers, while our agents can escalate complex cases that require human intervention.

3. **Data-Driven Customer Insights**: GenAI can analyze vast amounts of customer data, identifying patterns and trends that may not be immediately apparent to human analysts. For example, we can use GenAI to analyze customer feedback, social media sentiment, and purchase history to identify areas where we can improve our products and services, and create targeted marketing campaigns.

4. **Automated Lead Qualification**: GenAI can analyze incoming lead data, identifying key characteristics that indicate a high-quality lead. For instance, we can use GenAI to analyze lead data, such as company size, job title, and industry, to determine whether the lead is a good fit for our product or service. This can help our sales team focus on the most promising leads and reduce the time spent on unqualified opportunities.

These are just a few examples of how GenAI can transform our sales and customer engagement processes. I'd love to hear your thoughts and ideas on how we can leverage GenAI to drive business growth.


--- End Demonstration for this parameter setting ---


In [26]:
# Example 5.2: Moderate presence penalty (encourages covering a broader range of distinct action items)
generate_and_display(
    common_prompt_pres_penalty,
    "presence_penalty", 1.0, # Increased for more impact
    temperature=0.7,
)


--- Demonstrating presence_penalty=1.0 ---
Prompt: "
You are leading a quick internal meeting discussing how Generative AI can significantly improve our current sales and customer engagement processes. 
Briefly outline 4 distinct areas or strategies where GenAI could have a major impact, providing a brief example for each.
"
Other consistent parameters: {'temperature': 0.7}

--- Run 1 ---
Generated Text:


Here are 4 distinct areas where Generative AI (GenAI) can have a significant impact on our sales and customer engagement processes, along with brief examples:

**1. Personalized Content Generation**

* Example: Our marketing team creates personalized product recommendations for customers based on their browsing history and purchase behavior. GenAI-powered content generation tools can analyze customer data and generate customized product suggestions, increasing the likelihood of conversion.
* Impact: Increased sales revenue through targeted product recommendations.

**2. Automated Chatbot Conversations**

* Example: Our support team uses a GenAI-powered chatbot to answer frequently asked questions and route complex issues to human support agents. The chatbot can analyze customer conversation patterns and provide personalized responses, improving the overall customer experience.
* Impact: Reduced support queries and faster resolution times for complex issues.

**3. Data-Driven Sales Scripting**

* Example: Our sales team uses GenAI-powered script generation tools to create personalized sales conversations based on customer demographics and purchase history. The scripts can be fine-tuned to optimize conversion rates and improve sales outcomes.
* Impact: Improved sales performance through more effective and targeted conversations with customers.

**4. Predictive Lead Scoring**

* Example: Our marketing team uses GenAI-powered lead scoring models to predict the likelihood of a customer converting into a sale. The models analyze customer behavior, demographic data, and firmographic information to provide accurate and actionable insights.
* Impact: Increased revenue through targeted lead nurturing and prioritization.

These are just a few examples of how GenAI can be applied to improve sales and customer engagement processes. By leveraging these technologies, we can unlock significant efficiency gains, improved customer experiences, and increased revenue growth.


--- End Demonstration for this parameter setting ---


In [27]:
# Example 5.3: High presence penalty (strong encouragement for new ideas, potentially disjointed)
generate_and_display(
    common_prompt_pres_penalty,
    "presence_penalty", 2.0, # Max penalty for strong effect
    temperature=0.7,
)


--- Demonstrating presence_penalty=2.0 ---
Prompt: "
You are leading a quick internal meeting discussing how Generative AI can significantly improve our current sales and customer engagement processes. 
Briefly outline 4 distinct areas or strategies where GenAI could have a major impact, providing a brief example for each.
"
Other consistent parameters: {'temperature': 0.7}

--- Run 1 ---
Generated Text:


Here are 4 distinct areas where Generative AI (GenAI) can have a significant impact on our sales and customer engagement processes:

**1. Personalized Sales Content Generation**

* Using GenAI to generate personalized sales content, such as:
	+ Customized product descriptions for e-commerce platforms
	+ Tailored sales scripts for specific customers or industries
	+ Automated email templates for follow-up communications

Example: Our marketing team uses GenAI to create personalized product descriptions for our e-commerce platform. The AI tool analyzes customer data and preferences, generating unique descriptions that increase conversion rates by 20%.

**2. Chatbot-Powered Customer Engagement**

* Leveraging GenAI to power chatbots that:
	+ Provide instant answers to common customer questions
	+ Offer personalized recommendations based on user behavior
	+ Help customers with simple issues, freeing up human support agents for complex cases

Example: Our customer service team uses a GenAI-powered chatbot to assist with basic queries, such as order status and return policies. The chatbot reduces response times by 30% and allows human agents to focus on more complex issues.

**3. Predictive Lead Scoring and Qualification**

* Using GenAI to analyze large datasets and predict lead behavior, enabling:
	+ Automated lead scoring and qualification
	+ Priority assignment of leads to sales teams
	+ Identification of high-potential leads

Example: Our sales team uses a GenAI-powered lead scoring system that analyzes customer data and predicts likely conversion rates. This allows us to prioritize our efforts on the most promising leads, resulting in a 25% increase in qualified opportunities.

**4. Automated Sales Email and Outreach**

* Utilizing GenAI to generate and send personalized sales emails, including:
	+ Customized subject lines and body copy
	+ Personalized follow-up emails based on customer interactions
	+ Automated email sequences for nurturing leads

Example: Our sales team uses a GenAI-powered email generator to craft personalized subject lines and body copy for their outreach campaigns. This results in a 15% increase in response rates and a 20% reduction in response time.

These are just a few examples of how Generative AI can enhance our sales and customer engagement processes. I believe this technology has the potential to significantly improve our efficiency, effectiveness, and overall customer experience.


--- End Demonstration for this parameter setting ---


### 6. Stop

**Commercial Use Cases**: Ensuring chatbot responses don't continue past a natural break, extracting specific data blocks (e.g., JSON), generating bulleted lists that don't overextend, completing fill-in-the-blank forms.
garding:"

**What to Observe**: The model's generation will immediately halt once any of the provided stop sequences appear in its output.

In [28]:
# Example 6.1: Stopping at newline (for single-line completions)
common_prompt_stop = """
Draft a quick customer service response template:\nThank you for contacting TrueFoundry. We received your inquiry regarding:"""
generate_and_display(
    common_prompt_stop,
    "stop", ["\n"],
    temperature=0.5
)


--- Demonstrating stop=['\n'] ---
Prompt: "
Draft a quick customer service response template:
Thank you for contacting TrueFoundry. We received your inquiry regarding:"
Other consistent parameters: {'temperature': 0.5}

--- Run 1 ---
Generated Text:


Here's a quick customer service response template:


--- End Demonstration for this parameter setting ---


In [29]:
# Example 6.2: Stopping at a specific closing phrase (for structured email generation)
common_prompt_email = """Draft a professional email to a client regarding their contract status. 
Subject: Your Contract #12345 Update\nDear [Client Name],\n\nYour recent contract #12345 has been:"""
generate_and_display(
    common_prompt_email,
    "stop", ["Best regards,"], # Model will stop before adding the closing
    temperature=0.5
)


--- Demonstrating stop=['Best regards,'] ---
Prompt: "Draft a professional email to a client regarding their contract status. 
Subject: Your Contract #12345 Update
Dear [Client Name],

Your recent contract #12345 has been:"
Other consistent parameters: {'temperature': 0.5}

--- Run 1 ---
Generated Text:


Here's a draft of the email:

Subject: Your Contract #12345 Update

Dear [Client Name],

Re: Contract #12345

I am writing to provide you with an update on the status of your contract, reference number #12345. As of our latest review, we are pleased to inform you that your contract has been [insert status, e.g. "approved and finalized", "under review", "awaiting signature", etc.].

Please find the current details of the contract below for your reference:

* Contract Reference: #12345
* Status: [Insert status]
* Next Steps: [Insert next steps or any additional information]

If you have any questions or concerns regarding your contract, please do not hesitate to contact us. We are committed to ensuring a smooth and successful partnership and are available to address any matters that may arise.

Thank you for your continued partnership.


--- End Demonstration for this parameter setting ---


In [30]:
# Example 6.3: Stopping after a list item (for controlled list generation)
common_prompt_list = "List 3 benefits of our AI gateway:\n1. AI Governance\n2. Efficient budgeting\n3."
generate_and_display(
    common_prompt_list,
    "stop", ["4."], # Stop before generating the 4th item (assuming it would start with 4.)
    temperature=0.5
)


--- Demonstrating stop=['4.'] ---
Prompt: "List 3 benefits of our AI gateway:
1. AI Governance
2. Efficient budgeting
3."
Other consistent parameters: {'temperature': 0.5}

--- Run 1 ---
Generated Text:


Here are three benefits of your AI gateway:

1. **AI Governance**: Our AI gateway provides a centralized platform for managing and governing AI-related projects, data, and models, ensuring transparency, accountability, and compliance with regulatory requirements.
2. **Efficient Budgeting**: The AI gateway enables organizations to allocate resources more effectively by providing real-time analytics and insights on AI-related expenses, allowing for more informed budgeting decisions and reduced costs.
3. **Data Integration and Standardization**: The AI gateway facilitates seamless integration of data from various sources, ensuring that data is standardized, consistent, and accurate, which is essential for building a robust and reliable AI system.


--- End Demonstration for this parameter setting ---


### 7. Seed

**Commercial Use Cases**: Reproducible content for A/B testing (ensuring differences are due to prompt, not randomness), consistent chatbot behavior for quality assurance, debugging prompt engineering, generating consistent test data.

**What to Observe**:

- **Same** seed: The outputs for the multiple runs should be identical or extremely close.
- **Different** seed: The outputs, while still relevant to the prompt, should be distinct from each other across different seeds.

In [31]:
common_prompt_seed = "Generate a marketing slogan for a new eco-friendly cleaning product. Only return the slogan."

In [32]:
# Example 7.1: Same seed, same prompt, same parameters (for reproducible slogan generation for testing)
print("\n--- Testing reproducibility with the SAME seed (42) ---")
generate_and_display(
    common_prompt_seed,
    "seed", 42,
    num_runs=3, # Run multiple times to show consistency
    temperature=0.7,
    max_tokens=20
)


--- Testing reproducibility with the SAME seed (42) ---

--- Demonstrating seed=42 ---
Prompt: "Generate a marketing slogan for a new eco-friendly cleaning product. Only return the slogan."
Other consistent parameters: {'temperature': 0.7, 'max_tokens': 20}

--- Run 1 ---
Generated Text:


"Clean with a Clear Conscience"


--- Run 2 ---
Generated Text:


"Clean with a Clear Conscience"


--- Run 3 ---
Generated Text:


"Clean with a Clear Conscience"


--- End Demonstration for this parameter setting ---


In [33]:
# Example 7.2: Different seeds, same prompt, same parameters (to get diverse options for review)
print("\n--- Testing different seeds for DIVERSE slogan options ---")
generate_and_display(
    common_prompt_seed,
    "seed", 101,
    num_runs=1, # One run for each distinct seed is enough to show variety
    temperature=0.7,
    max_tokens=20
)


--- Testing different seeds for DIVERSE slogan options ---

--- Demonstrating seed=101 ---
Prompt: "Generate a marketing slogan for a new eco-friendly cleaning product. Only return the slogan."
Other consistent parameters: {'temperature': 0.7, 'max_tokens': 20}

--- Run 1 ---
Generated Text:


"Sparkle with a Clear Conscience"


--- End Demonstration for this parameter setting ---


In [34]:
# Example 7.3: Different seeds, same prompt, same parameters (to get diverse options for review)
generate_and_display(
    common_prompt_seed,
    "seed", 202,
    num_runs=1,
    temperature=0.7,
    max_tokens=20
)


--- Demonstrating seed=202 ---
Prompt: "Generate a marketing slogan for a new eco-friendly cleaning product. Only return the slogan."
Other consistent parameters: {'temperature': 0.7, 'max_tokens': 20}

--- Run 1 ---
Generated Text:


"Clean with a Clear Conscience"


--- End Demonstration for this parameter setting ---


In [35]:
# Example 7.4: Different seeds, same prompt, same parameters (to get diverse options for review)
generate_and_display(
    common_prompt_seed,
    "seed", 303,
    num_runs=1,
    temperature=0.7,
    max_tokens=20
)


--- Demonstrating seed=303 ---
Prompt: "Generate a marketing slogan for a new eco-friendly cleaning product. Only return the slogan."
Other consistent parameters: {'temperature': 0.7, 'max_tokens': 20}

--- Run 1 ---
Generated Text:


"Clean with a Clear Conscience"


--- End Demonstration for this parameter setting ---
